# Personal Expense Data Cleaning

This notebook consolidates raw expense CSV files, standardizes fields, cleans values, and exports a final cleaned dataset.

## 1) Inspect Raw Input Files

Load each source CSV and preview sample rows.

In [88]:
import pandas as pd
import glob

csv_files = glob.glob("./data/*.csv")

for file in csv_files:
    print("\n==============================")
    print("📄 File:", file)
    df = pd.read_csv(file)
    display(df.head())


📄 File: ./data/expense_data_1.csv


,Date,Account,Category,Subcategory,Note,INR,Income/Expense,Note.1,Amount,Currency,Account.1
0,3/2/2022 10:11,CUB - online payment,Food,NaN,Brownie,50.0,Expense,NaN,50.0,INR,50.0
1,3/2/2022 10:11,CUB - online payment,Other,NaN,To lended people,300.0,Expense,NaN,300.0,INR,300.0
2,3/1/2022 19:50,CUB - online payment,Food,NaN,Dinner,78.0,Expense,NaN,78.0,INR,78.0
3,3/1/2022 18:56,CUB - online payment,Transportation,NaN,Metro,30.0,Expense,NaN,30.0,INR,30.0
4,3/1/2022 18:22,CUB - online payment,Food,NaN,Snacks,67.0,Expense,NaN,67.0,INR,67.0



📄 File: ./data/expenses_income_summary.csv


,Date,title,category,account,amount,currency,type,transfer-amount,transfer-currency,to-account,receive-amount,receive-currency,description,due-date,id
0,2024-08-11 13:56:59.652,Karthik,Bills & Fees,Savings Bank,45.00,INR,EXPENSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74e78631-db14-4495-bfb9-85546b0bd2fe
1,2024-08-10 16:09:55.986,Juice,Food & Drinks,Cash,40.00,INR,EXPENSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65e12e62-9f63-4c6c-b452-6c7b42fbfb7f
2,2024-08-09 10:25:21.618,Tire,Transport,Cash,10.00,INR,EXPENSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9ecd93bd-a835-4263-86e2-99fea475fa37
3,2024-08-07 03:57:24.944,Baba,Bills & Fees,Savings Bank,200.00,INR,EXPENSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,00d39b2c-e722-485a-85ca-28f6506dc674
4,2024-08-04 13:09:08.452,Reward,Bills & Fees,Salary Bank,4.00,INR,INCOME,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3861d205-3245-4926-ad69-4491b0bff547



📄 File: ./data/Personal_Finance_Dataset.csv


,Date,Transaction Description,Category,Amount,Type
0,2020-01-02,Score each.,Food & Drink,1485.69,Expense
1,2020-01-02,Quality throughout.,Utilities,1475.58,Expense
2,2020-01-04,Instead ahead despite measure ago.,Rent,1185.08,Expense
3,2020-01-05,Information last everything thank serve.,Investment,2291.00,Income
4,2020-01-13,Future choice whatever from.,Food & Drink,1126.88,Expense


## 2) Combine and Standardize Source Data

Merge all CSV files into one dataframe and align key schema differences.

In [89]:
import pandas as pd
import glob

csv_files = glob.glob("./data/*.csv")

dfs = []

for file in csv_files:
    df = pd.read_csv(file)

    # normalize column names (case-insensitive)
    df.columns = [col.strip().lower() for col in df.columns]

    # if category exists in any form, rename to "categories"
    if "category" in df.columns:
        df = df.rename(columns={"category": "categories"})
    if "categories" not in df.columns:
        df["categories"] = None  # create empty if missing

    df["source_file"] = file  # optional: track file source
    dfs.append(df)

# combine all dataframes
combined_df = pd.concat(dfs, ignore_index=True, sort=False)

print("✅ Combined shape:", combined_df.shape)
combined_df.head()

✅ Combined shape: (2932, 23)


,date,account,categories,subcategory,note,inr,income/expense,note.1,amount,currency,...,type,transfer-amount,transfer-currency,to-account,receive-amount,receive-currency,description,due-date,id,transaction description
0,3/2/2022 10:11,CUB - online payment,Food,NaN,Brownie,50.0,Expense,NaN,50.0,INR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3/2/2022 10:11,CUB - online payment,Other,NaN,To lended people,300.0,Expense,NaN,300.0,INR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3/1/2022 19:50,CUB - online payment,Food,NaN,Dinner,78.0,Expense,NaN,78.0,INR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3/1/2022 18:56,CUB - online payment,Transportation,NaN,Metro,30.0,Expense,NaN,30.0,INR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3/1/2022 18:22,CUB - online payment,Food,NaN,Snacks,67.0,Expense,NaN,67.0,INR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3) Remove Unused Columns

Drop extra fields not needed for downstream analysis.

In [90]:
cols_to_remove = ["date", 
                  "subcategory", 
                  "note.1", 
                  "account", 
                  "transfer-amount", 
                  "transfer-currency", 
                  "to-account", 
                  "receive-amount", 
                  "receive-currency", 
                  "description", 
                  "due-date", 
                  "id",	
                  "transaction description",
                  "account.1",
                  "title",
                  "source_file",
                  "inr"
]

combined_df = combined_df.drop(
    columns=[c for c in cols_to_remove if c in combined_df.columns],
    errors="ignore"
)

In [91]:
combined_df.head()

,categories,note,income/expense,amount,currency,type
0,Food,Brownie,Expense,50.0,INR,NaN
1,Other,To lended people,Expense,300.0,INR,NaN
2,Food,Dinner,Expense,78.0,INR,NaN
3,Transportation,Metro,Expense,30.0,INR,NaN
4,Food,Snacks,Expense,67.0,INR,NaN


## 4) Normalize Transaction Type Fields

Unify type labels and consolidate overlapping columns.

In [92]:
# normalize column names first (important)
combined_df.columns = [c.strip().lower() for c in combined_df.columns]

# merge "income/expense" and "type" into one column called "type"
combined_df["type"] = combined_df["type"].fillna(combined_df["income/expense"])

# drop the old column
combined_df = combined_df.drop(columns=["income/expense"], errors="ignore")

combined_df.head()

,categories,note,amount,currency,type
0,Food,Brownie,50.0,INR,Expense
1,Other,To lended people,300.0,INR,Expense
2,Food,Dinner,78.0,INR,Expense
3,Transportation,Metro,30.0,INR,Expense
4,Food,Snacks,67.0,INR,Expense


## 5) Clean Currency and Convert Amounts

Fill missing currency values, convert INR amounts to USD, and standardize currency output.

In [93]:
empty_count = combined_df['currency'].isna().sum()

print(f"Number of empty values in 'currency' column: {empty_count}")

Number of empty values in 'currency' column: 1500


In [94]:
# Replace NaN or empty strings with 'USD'
combined_df['currency'] = combined_df['currency'].replace('', pd.NA).fillna('USD')

In [95]:
empty_count = combined_df['currency'].isna().sum()

print(f"Number of empty values in 'currency' column: {empty_count}")

Number of empty values in 'currency' column: 0


In [96]:
# Ensure amount is numeric
combined_df['amount'] = pd.to_numeric(combined_df['amount'], errors='coerce')

# Exchange rate
inr_to_usd = 0.012

# Convert INR to USD
combined_df.loc[combined_df['currency'] == 'INR', 'amount'] = combined_df['amount'] * inr_to_usd

# Keep only 2 decimal places
combined_df['amount'] = combined_df['amount'].round(2)

# Replace all currency with USD
combined_df['currency'] = 'USD'

In [97]:
combined_df

,categories,note,amount,currency,type
0,Food,Brownie,0.60,USD,Expense
1,Other,To lended people,3.60,USD,Expense
2,Food,Dinner,0.94,USD,Expense
3,Transportation,Metro,0.36,USD,Expense
4,Food,Snacks,0.80,USD,Expense
...,...,...,...,...,...
2927,Rent,NaN,514.09,USD,Expense
2928,Entertainment,NaN,727.25,USD,Expense
2929,Investment,NaN,1425.00,USD,Income
2930,Shopping,NaN,655.78,USD,Expense


## 6) Normalize Categories

Review category values, inspect missing categories, and map inconsistent labels.

In [98]:
unique_categories = combined_df['categories'].unique()
print(unique_categories)

<StringArray>
[            'Food',            'Other',   'Transportation',
      'Social Life',        'Household',          'Apparel',
        'Education',           'Salary',        'Allowance',
 'Self-development',           'Beauty',             'Gift',
       'Petty cash',     'Bills & Fees',    'Food & Drinks',
        'Transport',                nan,     'Food & Drink',
        'Utilities',             'Rent',       'Investment',
         'Shopping',    'Entertainment', 'Health & Fitness',
           'Travel']
Length: 25, dtype: str


In [99]:
# Get rows where 'categories' is NaN
nan_rows = combined_df[combined_df['categories'].isna()]

print(nan_rows)

     categories note  amount currency     type
285         NaN  NaN    0.89      USD  EXPENSE
286         NaN  NaN    1.22      USD  EXPENSE
292         NaN  NaN    0.00      USD   INCOME
295         NaN  NaN    0.04      USD  EXPENSE
299         NaN  NaN    0.49      USD   INCOME
...         ...  ...     ...      ...      ...
1427        NaN  NaN    0.08      USD   INCOME
1428        NaN  NaN    3.60      USD   INCOME
1429        NaN  NaN    1.67      USD   INCOME
1430        NaN  NaN     NaN      USD   INCOME
1431        NaN  NaN    3.00      USD   INCOME

[161 rows x 5 columns]


In [100]:
# Define mapping for category normalization
category_mapping = {
    # Merge food categories
    'food': 'Food & Drink',
    'food & drinks': 'Food & Drink',
    'food & drink': 'Food & Drink',
    
    # Merge transport/travel categories
    'transport': 'Transportation',
    'transportation': 'Transportation',
    'travel': 'Transportation'
}

# Normalize categories: lowercase + strip + map
combined_df['categories'] = (combined_df['categories']
                             .astype(str)
                             .str.strip()
                             .str.lower()
                             .map(category_mapping)
                             .fillna(combined_df['categories']))

# Optional: capitalize first letters for consistency
combined_df['categories'] = combined_df['categories'].str.title()

# Check unique categories after normalization
print(combined_df['categories'].unique())

<StringArray>
[    'Food & Drink',            'Other',   'Transportation',
      'Social Life',        'Household',          'Apparel',
        'Education',           'Salary',        'Allowance',
 'Self-Development',           'Beauty',             'Gift',
       'Petty Cash',     'Bills & Fees',                nan,
        'Utilities',             'Rent',       'Investment',
         'Shopping',    'Entertainment', 'Health & Fitness']
Length: 21, dtype: str


In [101]:
combined_df

,categories,note,amount,currency,type
0,Food & Drink,Brownie,0.60,USD,Expense
1,Other,To lended people,3.60,USD,Expense
2,Food & Drink,Dinner,0.94,USD,Expense
3,Transportation,Metro,0.36,USD,Expense
4,Food & Drink,Snacks,0.80,USD,Expense
...,...,...,...,...,...
2927,Rent,NaN,514.09,USD,Expense
2928,Entertainment,NaN,727.25,USD,Expense
2929,Investment,NaN,1425.00,USD,Income
2930,Shopping,NaN,655.78,USD,Expense


## 7) Validate Cleaned Data

Inspect unique values across columns and standardize final type formatting.

In [102]:
# Loop through each column and print unique values
for col in combined_df.columns:
    unique_vals = combined_df[col].unique()
    print(f"Unique values in column '{col}':")
    print(unique_vals)
    print("-" * 50)

Unique values in column 'categories':
<StringArray>
[    'Food & Drink',            'Other',   'Transportation',
      'Social Life',        'Household',          'Apparel',
        'Education',           'Salary',        'Allowance',
 'Self-Development',           'Beauty',             'Gift',
       'Petty Cash',     'Bills & Fees',                nan,
        'Utilities',             'Rent',       'Investment',
         'Shopping',    'Entertainment', 'Health & Fitness']
Length: 21, dtype: str
--------------------------------------------------
Unique values in column 'note':
<StringArray>
[            'Brownie',    'To lended people',              'Dinner',
               'Metro',              'Snacks',          'From vicky',
            'From dad',               'Pizza',         'From kumara',
               'Lunch',
 ...
                'Mask',           'Ice cream',            'Vadapaav',
           'Showergel',    'Bharath birthday',              'Zomato',
 'Dinner with gowdham'

In [103]:
# Normalize 'type' column to title case (or upper/lower)
combined_df['type'] = combined_df['type'].str.strip().str.title()  # 'Expense', 'Income', 'Transfer'

# Check unique values again
print(combined_df['type'].unique())

<StringArray>
['Expense', 'Income', 'Transfer']
Length: 3, dtype: str


## 8) Export Final Cleaned Dataset

Save the cleaned dataframe to a CSV file for downstream modeling and analysis.

In [104]:
# Save the cleaned DataFrame to a CSV file
combined_df.to_csv("cleaned_budget_data.csv", index=False)

print("✅ File saved as 'cleaned_budget_data.csv'")

✅ File saved as 'cleaned_budget_data.csv'
